# Pipeline gợi ý việc làm IT — chạy từng bước

Notebook này chạy **full luồng** trong một project:

`Crawl → Preprocess → Features → K-Means → KNN → Evaluate → Demo CV`

Mặc định **không cào lại** website (`SKIP_CRAWL = True`) vì repo đã có `data/raw/combined.jsonl` (~1711 JD). Đặt `SKIP_CRAWL = False` khi muốn thêm tin mới (cần Google Chrome).

Kernel: Python 3.10–3.12, cài `requirements.txt`. Cell setup tự `chdir` vào `IT_Job_Recommendation`.

## 0. Setup

In [ ]:
from pathlib import Path
import os, sys, json, warnings

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = Path.cwd().parent if (Path.cwd().parent / "config.yaml").exists() else ROOT
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

SKIP_CRAWL = True              # True: dùng combined.jsonl có sẵn
SKIP_TRAIN_IF_EXISTS = False   # True: bỏ qua fit nếu đã có model
CRAWL_MAX_PAGES = 1
CRAWL_MAX_JOBS = 5

In [ ]:
from src.paths import (
    COMBINED_JSONL, JOBS_CLEAN, JOBS_FEATURED, JOBS_CLUSTERED, JOBS_PKL,
    JOB_VECTORS, JOB_VECTORS_REDUCED, PIPELINE_PATH, SPACE_PATH,
    KMEANS_PATH, KNN_PATH, META_PATH, SAMPLES_DIR,
    artifacts_ready, load_config, ensure_dirs,
)
import pandas as pd
import numpy as np
import joblib

ensure_dirs()
cfg = load_config()
print("artifacts_ready =", artifacts_ready())
print("combined.jsonl exists =", COMBINED_JSONL.exists())
print("kmeans.k_max =", cfg["kmeans"]["k_max"], "| space =", cfg["space"])

## 1. Crawl JD (ITviec + TopCV)

Selenium listing → trang chi tiết. Ghi `data/raw/combined.jsonl`. Bỏ qua nếu đã có data.

In [ ]:
from src.crawl.pipeline import run_crawl

if SKIP_CRAWL:
    if not COMBINED_JSONL.exists():
        raise FileNotFoundError("Thiếu combined.jsonl — đặt SKIP_CRAWL = False hoặc copy file raw.")
    print(f"SKIP_CRAWL: {COMBINED_JSONL} ({COMBINED_JSONL.stat().st_size / 1e6:.1f} MB)")
else:
    run_crawl(
        sources=["itviec", "topcv"],
        root=ROOT,
        max_pages=CRAWL_MAX_PAGES,
        max_jobs=CRAWL_MAX_JOBS,
        workers=2,
    )
    print("Crawl xong →", COMBINED_JSONL)

In [ ]:
from src.io import load_raw_jobs

raw = load_raw_jobs(COMBINED_JSONL)
print("Số JD thô:", len(raw))
print("Nguồn:", pd.Series([r.get("source") for r in raw]).value_counts().to_dict())
pd.DataFrame(raw[:3])[["title", "company", "source", "location_raw"]]

## 2. Preprocess

Làm sạch chữ, parse lương / kinh nghiệm / cấp / địa điểm, gán `role_family` (nhãn yếu từ title). Output: `data/interim/jobs_clean.pkl`.

In [ ]:
from scripts.preprocess import run_preprocess

if SKIP_TRAIN_IF_EXISTS and JOBS_CLEAN.exists():
    df_clean = pd.read_pickle(JOBS_CLEAN)
    print("Load sẵn", JOBS_CLEAN, "→", len(df_clean), "JD")
else:
    df_clean = run_preprocess()

df_clean[["title", "company", "role_family", "level_norm", "location_norm", "years_mid"]].head()

In [ ]:
print("role_family:\n", df_clean["role_family"].value_counts().to_string())
print("\nlocation:\n", df_clean["location_norm"].value_counts().to_string())
print("\nlevel:\n", df_clean["level_norm"].value_counts().to_string())

## 3. Features

Skill TF-IDF (80%) + SVD text yêu cầu (10%) + năm/cấp (10%) → vector L2.
Output: `models/job_vectors.npy`, `models/feature_pipeline.joblib`.

Bước này chậm hơn (quét từ điển skill trên từng JD).

In [ ]:
from scripts.build_features import run_features

if SKIP_TRAIN_IF_EXISTS and JOB_VECTORS.exists() and PIPELINE_PATH.exists():
    df_feat = pd.read_pickle(JOBS_FEATURED)
    X = np.load(JOB_VECTORS)
    pipe = joblib.load(PIPELINE_PATH)
    print("Load sẵn vectors", X.shape)
else:
    df_feat, X, pipe = run_features()

print("jobs × dim =", X.shape)
print("skill/text/struct dim =", pipe.skill_dim, pipe.text_dim, pipe.struct_dim)
print("avg skills/job =", float(np.mean([len(s or []) for s in df_feat["skills_norm"]])))

## 4. K-Means

`StandardScaler → PCA(50) → L2`, chọn K = argmax silhouette (8…14). Tên cụm gán theo top skill. PCA 2D chỉ để vẽ.

In [ ]:
from scripts.train_kmeans import run_kmeans

if SKIP_TRAIN_IF_EXISTS and KMEANS_PATH.exists() and JOB_VECTORS_REDUCED.exists():
    df_cl = pd.read_pickle(JOBS_CLUSTERED)
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))
    print("Load sẵn K =", meta.get("best_k"), "jobs =", len(df_cl))
else:
    df_cl, meta = run_kmeans()

print("best_k =", meta["best_k"], "| dim", meta.get("n_features_raw"), "→", meta.get("n_features"))
pd.DataFrame(meta["k_search"])

In [ ]:
profiles = pd.DataFrame(meta["cluster_profiles"].values())
profiles[["cluster_id", "name", "size", "role_family", "example_title"]]

## 5. KNN Cosine

Chỉ mục láng giềng trên vector đã giảm chiều. Output: `models/knn.joblib`, `data/processed/jobs.pkl` — **từ đây mở được app**.

In [ ]:
from scripts.train_knn import run_knn

if SKIP_TRAIN_IF_EXISTS and KNN_PATH.exists() and JOBS_PKL.exists():
    knn = joblib.load(KNN_PATH)
    df_jobs = pd.read_pickle(JOBS_PKL)
    print("Load sẵn knn, jobs =", len(df_jobs))
else:
    knn, df_jobs = run_knn()

print(df_jobs.shape, "n_neighbors =", knn.n_neighbors)

## 6. Evaluate (tuỳ chọn)

Silhouette / Davies–Bouldin trên cụm; Precision@10 hold-out theo `role_family`; 3 persona cố định.

In [ ]:
from scripts.evaluate import run_evaluate

report = run_evaluate()
print("silhouette =", report["internal"].get("silhouette"))
print("davies_bouldin =", report["internal"].get("davies_bouldin"))
print("precision@10 =", report["retrieval"])
pd.DataFrame(report["personas"])[["name", "top_titles", "top_families"]]

## 7. Demo: đọc CV → gợi ý việc

Cùng hàm với Streamlit: parse CV mẫu → `embed_user` → `recommend`.

In [ ]:
from src.cv_parse.parser import parse_cv_path
from src.recommend.knn import embed_user, recommend

pipe = joblib.load(PIPELINE_PATH)
space = joblib.load(SPACE_PATH)
kmeans = joblib.load(KMEANS_PATH)
knn = joblib.load(KNN_PATH)
df_jobs = pd.read_pickle(JOBS_PKL)
meta = json.loads(META_PATH.read_text(encoding="utf-8"))

cv_path = SAMPLES_DIR / "cv_fresher_python.txt"
parsed = parse_cv_path(cv_path)
print("CV:", cv_path.name)
print("skills:", parsed["skills"])
print("years_mid:", parsed["years_mid"], "level:", parsed["level_norm"])

vec, canon = embed_user(
    pipe, space,
    parsed["skills"],
    parsed["years_mid"],
    parsed["level_ordinal"],
    extra_text=parsed["text"],
)
recs = recommend(
    user_vec=vec[0],
    user_skills=canon,
    user_years=parsed["years_mid"],
    user_level=parsed["level_ordinal"],
    df=df_jobs,
    knn=knn,
    kmeans=kmeans,
    top_n=8,
)
cluster_id = int(kmeans.predict(vec)[0])
cname = meta.get("cluster_profiles", {}).get(str(cluster_id), {}).get("name", cluster_id)
print("Nhóm nghề gần nhất:", cname)
recs[["rank", "score", "title", "company", "role_family", "overlap_skills", "missing_skills"]]

## Mở giao diện Streamlit

Trong terminal (đã activate venv, đứng tại `IT_Job_Recommendation`):

```powershell
streamlit run app/streamlit_app.py
```

Upload CV PDF/DOCX/TXT trên trang **Gợi ý từ CV**.